In [2]:
import requests
import time
import pandas as pd

# =========================
# CONFIG
# =========================
BASE_URL = "https://caching.graphql.imdb.com/"
OPERATION_NAME = "TitleReviewsRefine"
PAGE_SIZE = 25

HEADERS = {
    "accept": "application/graphql+json, application/json",
    "accept-language": "en-US,en;q=0.9",
    "content-type": "application/json",
    "origin": "https://www.imdb.com",
    "priority": "u=1, i",
    "user-agent": "Mozilla/5.0"
}

QUERY = """
query TitleReviewsRefine($const: ID!, $filter: ReviewsFilter, $first: Int!, $sort: ReviewsSort, $after: ID) {
  title(id: $const) {
    reviews(filter: $filter, first: $first, sort: $sort, after: $after) {
      edges {
        node {
          id
          author {
            nickName
          }
          authorRating
          helpfulness {
            upVotes
            downVotes
          }
          submissionDate
          text {
            originalText {
              plainText
            }
          }
          summary {
            originalText
          }
        }
      }
      pageInfo {
        hasNextPage
        endCursor
      }
    }
  }
}
"""

In [3]:
# =========================
# MCU IDS
# =========================
mcu_ids = [

    # ===== Phase One =====
    "tt0371746",  # Iron Man (2008)
    "tt0800080",  # The Incredible Hulk (2008)
    "tt1228705",  # Iron Man 2 (2010)
    "tt0800369",  # Thor (2011)
    "tt0458339",  # Captain America: The First Avenger (2011)
    "tt0848228",  # The Avengers (2012)

    # ===== Phase Two =====
    "tt1300854",  # Iron Man 3 (2013)
    "tt1981115",  # Thor: The Dark World (2013)
    "tt1843866",  # Captain America: The Winter Soldier (2014)
    "tt2015381",  # Guardians of the Galaxy (2014)
    "tt2395427",  # Avengers: Age of Ultron (2015)
    "tt0478970",  # Ant-Man (2015)

    # ===== Phase Three =====
    "tt3498820",  # Captain America: Civil War (2016)
    "tt1211837",  # Doctor Strange (2016)
    "tt3896198",  # Guardians of the Galaxy Vol. 2 (2017)
    "tt2250912",  # Spider-Man: Homecoming (2017)
    "tt3501632",  # Thor: Ragnarok (2017)
    "tt1825683",  # Black Panther (2018)
    "tt4154756",  # Avengers: Infinity War (2018)
    "tt5095030",  # Ant-Man and the Wasp (2018)
    "tt4154796",  # Avengers: Endgame (2019)
    "tt6320628",  # Spider-Man: Far From Home (2019)
    "tt4154664",  # Captain Marvel (2019)

    # ===== Phase Four =====
    "tt3480822",  # Black Widow (2021)
    "tt9376612",  # Shang-Chi and the Legend of the Ten Rings (2021)
    "tt9032400",  # Eternals (2021)
    "tt10872600", # Spider-Man: No Way Home (2021)
    "tt9419884",  # Doctor Strange in the Multiverse of Madness (2022)
    "tt10648342", # Thor: Love and Thunder (2022)
    "tt9114286",  # Black Panther: Wakanda Forever (2022)

    # ===== Phase Five =====
    "tt10954600", # Ant-Man and the Wasp: Quantumania (2023)
    "tt6791350",  # Guardians of the Galaxy Vol. 3 (2023)
    "tt10676048", # The Marvels (2023)
    "tt6263850",  # Deadpool & Wolverine (2024)
    "tt14513804", # Captain America: Brave New World (2025)
    "tt20969586", # Thunderbolts* (2025)

    # ===== Phase Six =====
    "tt10676052", # The Fantastic Four: First Steps (2025)
]

# Optional: map IDs to movie names
movie_map = {
    "tt0371746": "Iron Man",
    "tt0800080": "The Incredible Hulk",
    "tt1228705": "Iron Man 2",
    "tt0800369": "Thor",
    "tt0458339": "Captain America: The First Avenger",
    "tt0848228": "The Avengers",
    "tt1300854": "Iron Man 3",
    "tt1981115": "Thor: The Dark World",
    "tt1843866": "Captain America: The Winter Soldier",
    "tt2015381": "Guardians of the Galaxy",
    "tt2395427": "Avengers: Age of Ultron",
    "tt0478970": "Ant-Man",
    "tt3498820": "Captain America: Civil War",
    "tt1211837": "Doctor Strange",
    "tt3896198": "Guardians of the Galaxy Vol. 2",
    "tt2250912": "Spider-Man: Homecoming",
    "tt3501632": "Thor: Ragnarok",
    "tt1825683": "Black Panther",
    "tt4154756": "Avengers: Infinity War",
    "tt5095030": "Ant-Man and the Wasp",
    "tt4154796": "Avengers: Endgame",
    "tt6320628": "Spider-Man: Far From Home",
    "tt4154664": "Captain Marvel",
    "tt3480822": "Black Widow",
    "tt9376612": "Shang-Chi and the Legend of the Ten Rings",
    "tt9032400": "Eternals",
    "tt10872600": "Spider-Man: No Way Home",
    "tt9419884": "Doctor Strange in the Multiverse of Madness",
    "tt10648342": "Thor: Love and Thunder",
    "tt9114286": "Black Panther: Wakanda Forever",
    "tt10954600": "Ant-Man and the Wasp: Quantumania",
    "tt6791350": "Guardians of the Galaxy Vol. 3",
    "tt10676048": "The Marvels",
    "tt6263850": "Deadpool & Wolverine",
    "tt14513804": "Captain America: Brave New World",
    "tt20969586": "Thunderbolts*",
    "tt10676052": "The Fantastic Four: First Steps",
}

In [4]:
# =========================
# HELPERS
# =========================
def get_variables(movie_id, after_cursor=None, sort_by="HELPFULNESS_SCORE", sort_order="DESC"):
    variables = {
        "const": movie_id,
        "filter": {},
        "first": PAGE_SIZE,
        "sort": {
            "by": sort_by,
            "order": sort_order
        }
    }
    if after_cursor:
        variables["after"] = after_cursor
    return variables


def fetch_page(movie_id, after_cursor=None, sort_by="HELPFULNESS_SCORE", sort_order="DESC", timeout=30):
    payload = {
        "query": QUERY,
        "operationName": OPERATION_NAME,
        "variables": get_variables(movie_id, after_cursor, sort_by, sort_order)
    }

    response = requests.post(
        BASE_URL,
        headers=HEADERS,
        json=payload,
        timeout=timeout
    )
    response.raise_for_status()
    return response.json()


def extract_reviews(data, movie_id):
    movie_title = movie_map.get(movie_id, None)

    edges = (
        data.get("data", {})
        .get("title", {})
        .get("reviews", {})
        .get("edges", [])
    )

    rows = []
    for edge in edges:
        node = edge.get("node", {}) or {}

        rows.append({
            "movie_id": movie_id,
            "movie_title": movie_title,
            "review_id": node.get("id"),
            "author": (node.get("author") or {}).get("nickName"),
            "author_rating": node.get("authorRating"),
            "upvotes": (node.get("helpfulness") or {}).get("upVotes"),
            "downvotes": (node.get("helpfulness") or {}).get("downVotes"),
            "submission_date": node.get("submissionDate"),
            "summary": (node.get("summary") or {}).get("originalText"),
            "content": ((node.get("text") or {}).get("originalText") or {}).get("plainText"),
        })

    return rows


def get_all_reviews_for_movie(movie_id, delay=0.75, max_pages=None, sort_by="HELPFULNESS_SCORE", sort_order="DESC"):
    all_reviews = []
    after_cursor = None
    page = 1

    while True:
        try:
            data = fetch_page(
                movie_id=movie_id,
                after_cursor=after_cursor,
                sort_by=sort_by,
                sort_order=sort_order
            )
        except Exception as e:
            print(f"[ERROR] {movie_id} page {page}: {e}")
            break

        if "errors" in data:
            print(f"[ERROR] IMDb returned errors for {movie_id} page {page}: {data['errors']}")
            break

        page_reviews = extract_reviews(data, movie_id)
        all_reviews.extend(page_reviews)

        reviews_obj = (
            data.get("data", {})
            .get("title", {})
            .get("reviews", {})
        )

        page_info = reviews_obj.get("pageInfo", {}) or {}
        has_next = page_info.get("hasNextPage", False)
        end_cursor = page_info.get("endCursor")

        print(f"{movie_map.get(movie_id, movie_id)} | page {page} | reviews collected so far: {len(all_reviews)}")

        if not has_next:
            break

        if max_pages is not None and page >= max_pages:
            break

        if not end_cursor:
            break

        after_cursor = end_cursor
        page += 1
        time.sleep(delay)

    return all_reviews


def clean_reviews_df(df):
    if df.empty:
        return df

    if "content" in df.columns:
        df["content"] = (
            df["content"]
            .fillna("")
            .astype(str)
            .str.replace("\n", " ", regex=False)
            .str.replace("\r", " ", regex=False)
            .str.strip()
        )

    if "summary" in df.columns:
        df["summary"] = (
            df["summary"]
            .fillna("")
            .astype(str)
            .str.replace("\n", " ", regex=False)
            .str.replace("\r", " ", regex=False)
            .str.strip()
        )

    df = df.drop_duplicates(subset=["review_id"]).reset_index(drop=True)
    df = df[df["content"].str.len() > 0].reset_index(drop=True)

    return df

In [6]:
# =========================
# TO DATAFRAME
# =========================
df_reviews = pd.DataFrame(all_reviews)
df_reviews = clean_reviews_df(df_reviews)

df_stats = pd.DataFrame(movie_stats).sort_values(
    by=["n_reviews", "movie_title"],
    ascending=[False, True]
).reset_index(drop=True)

print("\nTotal reviews:", len(df_reviews))


Total reviews: 90310


In [23]:
df_stats

,movie_id,movie_title,n_reviews
0,tt0478970,Ant-Man,25
1,tt5095030,Ant-Man and the Wasp,25
2,tt10954600,Ant-Man and the Wasp: Quantumania,25
3,tt2395427,Avengers: Age of Ultron,25
4,tt4154756,Avengers: Infinity War,25
5,tt1825683,Black Panther,25
6,tt9114286,Black Panther: Wakanda Forever,25
7,tt3480822,Black Widow,25
8,tt14513804,Captain America: Brave New World,25
9,tt3498820,Captain America: Civil War,25


In [7]:
# =========================
# SAVE
# =========================
df_reviews.to_csv("mcu_imdb_reviews.csv", index=False)
df_reviews.to_parquet("mcu_imdb_reviews.parquet", index=False)

df_stats.to_csv("mcu_review_counts.csv", index=False)

print("\nSaved:")
print("- mcu_imdb_reviews.csv")
print("- mcu_imdb_reviews.parquet")
print("- mcu_review_counts.csv")


Saved:
- mcu_imdb_reviews.csv
- mcu_imdb_reviews.parquet
- mcu_review_counts.csv
